# Part 2a: MT Uncertainties and Export to SimPEG and UBCGIF Convention

**WHAT THE NOTEBOOK DOES:**

1. Loads the sorted data file (n_freq, n_comp, n_loc)
2. Defines user-specified uncertainties
3. Uses the uncertainties to add Gaussian noise to the data
4. Converts and outputs true data, observed data and uncertainties in both SimPEG and UBCGIF convention

**Conversions:**

To go from original convention (Northing-Easting-Down and $+i\omega t$) to SimPEG (Easting-Northing-Up and $+i\omega t$):

* ZXX --> ZYY
* ZXY --> ZYX
* ZYX --> ZXY
* ZYY --> ZXX

To go from original convention (Northing-Easting-Down and $+i\omega t$) to UBCGIF (Northing-Easting-Down and $i\omega t$):

* Re[Zij] --> Re[Zij]
* -Im[Zij] --> Im[Zij]

In [1]:
from simpeg.electromagnetics import natural_source as nsem
from simpeg.utils import mkvc, ndgrid, plot2Ddata

# Basic Python functionality
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FormatStrFormatter

mpl.rcParams.update({"font.size": 14})

from ipywidgets import (
    interact,
    interactive,
    IntSlider,
    widget,
    FloatText,
    FloatSlider,
    fixed,
)

## Load the Data

In [2]:
in_dir = './part_1_outputs/'
out_dir = './part_2_outputs/'

if not os.path.exists(out_dir):
    os.mkdir(out_dir)

In [3]:
frequencies = np.load(in_dir + 'frequencies_mt.npy')
locations = np.load(in_dir + 'locations_mt.npy')
dtrue = np.load(in_dir + 'data_mt.npy')

n_freq = np.shape(dtrue)[0]
n_comp = np.shape(dtrue)[1]
n_loc = np.shape(dtrue)[2]

## Assign Uncertainties

* For off-diagonal impedances, do a percent uncertainty
* For diagonal impedances, do a fraction of the largest anomaly amplitude

In [4]:
pct = 0.05
frac = 0.1  # blanket uncert for diagonals

rx_type_list = ['ZXXR', 'ZXXI', 'ZXYR', 'ZXYI', 'ZYXR', 'ZYXR', 'ZYYR', 'ZYYI']

In [5]:
unc = np.zeros_like(dtrue)

for jj in range(n_freq):
    for kk in range(n_comp):
        
        d_temp = dtrue[jj, kk, :]
        
        if ('XX' in rx_type_list[kk]) or ('YY' in rx_type_list[kk]):
            unc[jj, kk, :] = frac * np.max(np.abs(d_temp))
        else:
            unc[jj, kk, :] = pct * np.abs(d_temp)

noise = unc * np.random.normal(size=(n_freq, n_comp, n_loc))
dobs = dtrue + noise

## Plot

In [6]:
mpl.rcParams.update({'font.size': 13})

d_list = [dtrue, dobs, noise]

def plot_data(f_ind, rx_ind):
    
    f_ind = f_ind - 1
    rx_ind = rx_ind - 1

    fig = plt.figure(figsize=(12, 4))

    ax1 = 3 * [None]
    ax2 = 3 *[None]
    norm = 3 * [None]
    cs = 3 * [None]
    cplot = 3 * [None]
    cbar = 3 * [None]
    cmap = 3 * [None]

    COUNT = 0

    for ii in range(3):  # true, obs, noise

        ax1[COUNT] = fig.add_axes([0.05+0.3*ii, 0.25, 0.25, 0.7])
        ax2[COUNT] = fig.add_axes([0.07+0.3*ii, 0.05, 0.21, 0.05])

        if ii == 0:
            data_temp = d_list[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dtrue' 
        elif ii == 1:
            data_temp = d_list[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dobs'
        if ii == 2:
            data_temp = d_list[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.RdBu_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'noise'

        norm[COUNT]= mpl.colors.Normalize(vmin=vmin, vmax=vmax)

        cplot[COUNT], ax1[COUNT] = plot2Ddata(
            locations[:, 0:2],
            data_temp,
            nx=200,
            ny=200,
            ax=ax1[COUNT],
            ncontour=200,
            contourOpts={"cmap": cmap[COUNT], "norm": norm[COUNT]},
        )

        ax1[COUNT].set_title(title + ": {}, {} Hz".format(rx_type_list[rx_ind], frequencies[f_ind]))
        ax1[COUNT].set_xlabel('Easting (m)')
        if ii == 0:
            ax1[COUNT].set_ylabel('Northing (m)')
        else:
            ax1[COUNT].set_yticks([])

        cbar[COUNT]= mpl.colorbar.ColorbarBase(ax2[COUNT], norm=norm[COUNT], orientation="horizontal", cmap=cmap[COUNT])
        cbar[COUNT].set_label('')
        ax2[COUNT].set_xticks([vmin, 0.5*(vmin+vmax), vmax])
        
        COUNT = COUNT + 1

def DataWidget():

    i = interact(
        plot_data,
        f_ind=IntSlider(
            min=1,
            max=n_freq,
            value=1,
            step=1,
            continuous_update=False,
            description="FREQID",
        ),
        rx_ind=IntSlider(
            min=1,
            max=n_comp,
            value=1,
            step=1,
            continuous_update=False,
            description="RXID",
        ),
    )
    
    return i

In [7]:
DataWidget()

interactive(children=(IntSlider(value=1, continuous_update=False, description='FREQID', max=35, min=1), IntSli…

<function __main__.plot_data(f_ind, rx_ind)>

## Convert and Output Data

### To SimPEG

In [8]:
dtrue_simpeg = dtrue[:, [6, 7, 4, 5, 2, 3, 0, 1], :]
dobs_simpeg = dobs[:, [6, 7, 4, 5, 2, 3, 0, 1], :]
unc_simpeg = unc[:, [6, 7, 4, 5, 2, 3, 0, 1], :]

In [9]:
np.save(out_dir + 'dtrue_mt_simpeg.npy', dtrue_simpeg)
np.save(out_dir + 'dobs_mt_simpeg.npy', dobs_simpeg)
np.save(out_dir + 'unc_mt_simpeg.npy', unc_simpeg)

### To UBCGIF

In [10]:
dtrue_ubcgif = dtrue.copy()
dtrue_ubcgif[:, 1::2, :] *= -1
dobs_ubcgif = dobs.copy()
dobs_ubcgif[:, 1::2, :] *= -1
unc_ubcgif = unc.copy()  # Uncertainties alwasy positive

In [11]:
np.save(out_dir + 'dtrue_mt_ubcgif.npy', dtrue_ubcgif)
np.save(out_dir + 'dobs_mt_ubcgif.npy', dobs_ubcgif)
np.save(out_dir + 'unc_mt_ubcgif.npy', unc_ubcgif)